# NB04 — Numba JIT Compilation

**Module 17: HPC, Parallel Computing, and Rust**

---

## 1. Motivation

NumPy vectorization eliminates Python loops — but only for operations that map onto NumPy's built-in ufuncs. When you need a custom recurrence (like filling a Smith-Waterman dynamic programming table), there is no NumPy ufunc for that. You either write a slow Python loop, or you use a compiled extension.

**Numba** lets you annotate an ordinary Python function with `@numba.jit` and have it compiled to native machine code the first time it is called. No C, no Cython, no separate compilation step — just a decorator.

---

## 2. Intuition

Numba uses **LLVM** (the same compiler backend as Clang) to compile Python functions. The first call to a JIT-compiled function incurs a **compilation overhead** (milliseconds to seconds depending on complexity). Every subsequent call runs at near-C speed.

`nopython=True` (also written as `@numba.njit`) forces Numba to compile without any Python fallback — if Numba can't compile a line, it errors rather than silently falling back to slow Python. Always use `nopython=True` for performance-critical code.

`numba.prange` is a drop-in replacement for `range` inside a JIT function that tells Numba the loop iterations are independent and can be parallelized across CPU cores (requires `parallel=True` in the decorator).

---

## 3. Biological background

**Smith-Waterman local alignment:** The canonical algorithm for finding the best local alignment between two sequences. It fills an $(M+1) \times (N+1)$ scoring matrix where each cell depends on its left, upper, and upper-left neighbors — a sequential recurrence that cannot be vectorized with NumPy's standard operations.

This is the prototypical example of a computation that requires a compiled loop — and that Numba handles beautifully.

**Sliding window GC content** is trivially vectorizable with NumPy (NB01), but JIT-compiling the explicit loop with Numba provides a useful baseline comparison: how close does Numba get to NumPy's optimized cumsum trick?

---

## 4. Mathematical explanation

### Smith-Waterman scoring matrix fill

Given sequences $a$ (length $M$) and $b$ (length $N$), fill matrix $H$ where:

$$H[i][j] = \max\begin{cases} 0 \\ H[i-1][j-1] + \text{score}(a[i], b[j]) \\ H[i-1][j] + \text{gap} \\ H[i][j-1] + \text{gap} \end{cases}$$

with `score(a, b) = match` if `a == b` else `mismatch`, and `gap` is a negative penalty.

The maximum entry in $H$ is the best local alignment score. Complexity: $O(MN)$, irreducibly sequential.

### Euclidean distance matrix

For $N$ points in $D$ dimensions, pairwise Euclidean distance:

$$d_{ij} = \|x_i - x_j\|_2 = \sqrt{\sum_{k=1}^{D} (x_{ik} - x_{jk})^2}$$

Can be vectorized with broadcasting, but for large $N$ the intermediate $(N, N, D)$ tensor causes memory pressure. A Numba JIT loop avoids this.

---

## 5. Computational explanation

```python
import numba

@numba.njit                      # nopython=True, no fallback
def fn(x):                       # compiled on first call
    ...

@numba.njit(parallel=True)       # enable prange parallelism
def fn_parallel(x):
    for i in numba.prange(len(x)):  # parallel range
        ...

@numba.vectorize(['float64(float64)'])  # ufunc: scalar → scalar, applied element-wise
def fn_vec(x):
    ...
```

**Warm vs cold timing:** Always time Numba functions twice. First call = compile + run (cold). Second+ calls = run only (warm). The warm time is what matters for performance comparisons.

---

## 6. Python implementation

In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt

try:
    import numba
    NUMBA_AVAILABLE = True
    print(f"Numba version: {numba.__version__}")
except ImportError:
    NUMBA_AVAILABLE = False
    print("Numba not installed — install with: pip install numba")
    print("The notebook will show the code patterns with fallback pure Python.")

np.random.seed(42)

In [ ]:
# --- (1) Sliding window GC content: three implementations ---

def sliding_gc_python(seq, w):
    """Python loop — baseline."""
    n = len(seq)
    result = np.empty(n - w + 1, dtype=np.float64)
    for i in range(n - w + 1):
        gc = 0
        for j in range(w):
            if seq[i + j] == 2 or seq[i + j] == 3:
                gc += 1
        result[i] = gc / w
    return result

def sliding_gc_numpy(seq, w):
    """NumPy cumsum trick — vectorized."""
    gc_bool = (seq == 2) | (seq == 3)
    cs = np.cumsum(gc_bool.astype(np.int32))
    cs = np.concatenate(([0], cs))
    return (cs[w:] - cs[:-w]) / w

if NUMBA_AVAILABLE:
    @numba.njit
    def sliding_gc_numba(seq, w):
        """Numba JIT loop — same algorithm as Python, compiled."""
        n = len(seq)
        result = np.empty(n - w + 1, dtype=numba.float64)
        for i in range(n - w + 1):
            gc = 0
            for j in range(w):
                if seq[i + j] == 2 or seq[i + j] == 3:
                    gc += 1
            result[i] = gc / w
        return result

seq_test = np.random.randint(0, 4, size=1000, dtype=np.int8)
w = 50

r_py = sliding_gc_python(seq_test, w)
r_np = sliding_gc_numpy(seq_test, w)
assert np.allclose(r_py, r_np), "NumPy disagrees with Python!"
print("NumPy correctness: PASSED")

if NUMBA_AVAILABLE:
    # Warmup (compilation)
    _ = sliding_gc_numba(seq_test, w)
    r_nb = sliding_gc_numba(seq_test, w)
    assert np.allclose(r_py, r_nb, atol=1e-6), "Numba disagrees!"
    print("Numba correctness: PASSED")

In [ ]:
# --- Timing across input sizes ---
sizes = [1_000, 10_000, 100_000, 1_000_000]
w = 100

times_py = []
times_np = []
times_nb = []

for sz in sizes:
    seq = np.random.randint(0, 4, size=sz, dtype=np.int8)
    
    # Python (skip large sizes)
    if sz <= 10_000:
        t0 = time.perf_counter()
        for _ in range(3): sliding_gc_python(seq, w)
        times_py.append((time.perf_counter() - t0) / 3)
    else:
        times_py.append(times_py[-1] * (sz / sizes[sizes.index(sz) - 1]))
    
    # NumPy
    t0 = time.perf_counter()
    for _ in range(10): sliding_gc_numpy(seq, w)
    times_np.append((time.perf_counter() - t0) / 10)
    
    # Numba (warm)
    if NUMBA_AVAILABLE:
        t0 = time.perf_counter()
        for _ in range(10): sliding_gc_numba(seq, w)
        times_nb.append((time.perf_counter() - t0) / 10)
    else:
        times_nb.append(None)

print(f"{'Size':>10} {'Python(s)':>12} {'NumPy(s)':>12} {'Numba(s)':>12}")
for sz, tp, tn, tnb in zip(sizes, times_py, times_np, times_nb):
    nb_str = f"{tnb:.6f}" if tnb is not None else "N/A"
    print(f"{sz:>10,} {tp:>12.6f} {tn:>12.6f} {nb_str:>12}")

In [ ]:
# --- (2) Smith-Waterman scoring matrix fill ---

def sw_fill_python(a, b, match=2, mismatch=-1, gap=-1):
    """Python loop implementation of Smith-Waterman matrix fill.
    Returns the score matrix H and the maximum alignment score.
    """
    M, N = len(a), len(b)
    H = np.zeros((M + 1, N + 1), dtype=np.int32)
    max_score = 0
    for i in range(1, M + 1):
        for j in range(1, N + 1):
            diag = H[i-1, j-1] + (match if a[i-1] == b[j-1] else mismatch)
            up   = H[i-1, j] + gap
            left = H[i, j-1] + gap
            H[i, j] = max(0, diag, up, left)
            if H[i, j] > max_score:
                max_score = H[i, j]
    return H, max_score

if NUMBA_AVAILABLE:
    @numba.njit
    def sw_fill_numba(a, b, match=2, mismatch=-1, gap=-1):
        M, N = len(a), len(b)
        H = np.zeros((M + 1, N + 1), dtype=numba.int32)
        max_score = 0
        for i in range(1, M + 1):
            for j in range(1, N + 1):
                diag = H[i-1, j-1] + (match if a[i-1] == b[j-1] else mismatch)
                up   = H[i-1, j] + gap
                left = H[i, j-1] + gap
                H[i, j] = max(0, diag, up, left)
                if H[i, j] > max_score:
                    max_score = H[i, j]
        return H, max_score

# Test sequences
a = np.array([0, 1, 2, 3, 0, 1, 2], dtype=np.int32)  # ATCGAT + C
b = np.array([1, 2, 3, 0, 1, 2, 3], dtype=np.int32)

H_py, score_py = sw_fill_python(a, b)
print(f"Smith-Waterman score (Python): {score_py}")
print(f"Score matrix (first 5x5):\n{H_py[:5, :5]}")

if NUMBA_AVAILABLE:
    _ = sw_fill_numba(a, b)  # warmup
    H_nb, score_nb = sw_fill_numba(a, b)
    assert score_py == score_nb, "SW scores disagree!"
    print(f"Smith-Waterman score (Numba): {score_nb}  [PASSED]")

# Benchmark on longer sequences
a_long = np.random.randint(0, 4, 500, dtype=np.int32)
b_long = np.random.randint(0, 4, 500, dtype=np.int32)

t0 = time.perf_counter()
for _ in range(5): sw_fill_python(a_long, b_long)
t_sw_py = (time.perf_counter() - t0) / 5

if NUMBA_AVAILABLE:
    t0 = time.perf_counter()
    for _ in range(100): sw_fill_numba(a_long, b_long)
    t_sw_nb = (time.perf_counter() - t0) / 100
    print(f"\nSW (L=500): Python={t_sw_py:.4f}s  Numba={t_sw_nb:.6f}s  Speedup={t_sw_py/t_sw_nb:.0f}x")

In [ ]:
# --- Cold vs warm compilation timing ---
if NUMBA_AVAILABLE:
    import numba as nb
    
    @nb.njit
    def fresh_function(x):
        """A fresh JIT function to demonstrate compile overhead."""
        total = 0.0
        for i in range(len(x)):
            total += x[i] ** 2
        return total
    
    test_input = np.random.randn(10_000)
    
    t0 = time.perf_counter()
    r_cold = fresh_function(test_input)  # first call: compile + run
    t_cold = time.perf_counter() - t0
    
    times_warm = []
    for _ in range(20):
        t0 = time.perf_counter()
        fresh_function(test_input)
        times_warm.append(time.perf_counter() - t0)
    t_warm = np.mean(times_warm)
    
    print(f"Cold call (compile + run): {t_cold:.4f} s")
    print(f"Warm call (run only):      {t_warm:.6f} s")
    print(f"Compilation overhead:      {t_cold - t_warm:.4f} s")
    print(f"Cold/warm ratio:           {t_cold/t_warm:.0f}x")
    print("\nKey insight: benchmark Numba only after the first (compile) call.")

## 7. Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel 1: Timing comparison across implementations and input sizes
ax = axes[0]
ax.loglog(sizes, times_py, 'o-', color='#e74c3c', label='Python loop', linewidth=2)
ax.loglog(sizes, times_np, 's-', color='#2980b9', label='NumPy cumsum', linewidth=2)
if any(t is not None for t in times_nb):
    ax.loglog(sizes, times_nb, '^-', color='#27ae60', label='Numba JIT (warm)', linewidth=2)
ax.set_xlabel('Sequence length', fontsize=11)
ax.set_ylabel('Wall time (s)', fontsize=11)
ax.set_title('Sliding GC Content:\nPython vs NumPy vs Numba', fontweight='bold')
ax.legend()
ax.grid(True, which='both', alpha=0.3)

# Panel 2: Cold vs warm compilation
ax = axes[1]
if NUMBA_AVAILABLE:
    categories = ['Cold\n(compile+run)', 'Warm\n(run only)', 'NumPy\nequivalent']
    # NumPy equivalent for same computation
    t0 = time.perf_counter()
    for _ in range(20):
        np.sum(test_input ** 2)
    t_np_equiv = (time.perf_counter() - t0) / 20
    
    vals = [t_cold, t_warm, t_np_equiv]
    colors = ['#e74c3c', '#27ae60', '#2980b9']
    bars = ax.bar(categories, vals, color=colors, edgecolor='black', alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v * 1.05,
                f'{v*1000:.2f}ms', ha='center', va='bottom', fontsize=9)
    ax.set_ylabel('Wall time (s)', fontsize=11)
    ax.set_title('Numba: Cold vs Warm Call\n(JIT compilation overhead)', fontweight='bold')
    ax.set_yscale('log')
    ax.grid(True, axis='y', alpha=0.3)
else:
    ax.text(0.5, 0.5, 'Numba not installed\npip install numba',
            ha='center', va='center', transform=ax.transAxes, fontsize=12)
    ax.set_title('Numba (not installed)', fontweight='bold')

plt.tight_layout()
plt.savefig('../datasets/nb04_numba.png', dpi=100, bbox_inches='tight')
plt.show()

## 8. Exercises

1. **Numba prange:** Write a Numba `@njit(parallel=True)` version of the Euclidean distance matrix computation that uses `numba.prange` for the outer loop. Compare to the `@njit` (single-threaded) version.

2. **Numba vectorize:** Write a `@numba.vectorize` ufunc `gc_indicator(base)` that returns 1.0 if `base == 2 or base == 3` else 0.0. Apply it to the integer-encoded sequence array and call `.mean()` on the result. Compare to the boolean mask NumPy approach from NB01.

3. **Type signatures:** Numba can be given explicit type signatures. Write `@numba.njit('float64[:](int8[:], int64)')` for `sliding_gc_numba`. What happens if you call it with `int32` input? How do you fix it?

4. **When Numba fails:** Try `@numba.njit` on a function that calls `Counter()` from `collections`. What error do you get? Why? How would you rewrite the k-mer counter to be JIT-compatible?

## 12. Reflection

*In your own words: When does Numba outperform NumPy? When does NumPy outperform Numba? What is the key trade-off between Numba and Rust (coming in NB07–NB09)?*

---

## Papers referenced

- Lam, S.K. et al. (2015). "Numba: A LLVM-based Python JIT compiler." *Proc. 2nd Workshop on the LLVM Compiler Infrastructure in HPC*. ACM.

## References

- Numba docs: https://numba.readthedocs.io/en/stable/
- Numba `@jit` vs `@njit` vs `@vectorize`: https://numba.readthedocs.io/en/stable/user/jit.html
- Smith-Waterman algorithm: https://en.wikipedia.org/wiki/Smith%E2%80%93Waterman_algorithm

## Future work / open questions

- Numba supports CUDA GPU kernels via `@numba.cuda.jit`. For the Smith-Waterman scoring matrix, how would you parallelize across anti-diagonals on a GPU?
- How does Numba's compilation cache work? Where are compiled bytecodes stored, and how do you invalidate them?